# LeetCode 1198 — Find Smallest Common Element in All Rows bằng Python

Link đề: https://leetcode.com/problems/find-smallest-common-element-in-all-rows/ · Độ khó: **Medium** · Chủ đề: Array, Hash Table, Binary Search, Matrix, Counting

> **Bài Premium — đề viết theo hiểu biết về bài, nên đối chiếu lại nếu có tài khoản Premium.** Đề được viết lại theo bản mô tả công khai (doocs); chữ ký hàm suy ra từ lời giải Python tham khảo.

Skeleton LeetCode:

```python
class Solution:
    def smallestCommonElement(self, mat: List[List[int]]) -> int:
        ...
```

Ghi chú: mỗi hàng **tăng nghiêm ngặt** (không có phần tử trùng trong một hàng) — chi tiết này cho phép **đếm** số hàng chứa mỗi giá trị chỉ bằng cách đếm số lần xuất hiện. Các lời giải: kiểm tra từng ứng viên của hàng đầu (tuyến tính hoặc **tìm kiếm nhị phân**), **đếm tần suất**, **giao tập hợp**, và **nhiều con trỏ** (m-way merge).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: tìm giá trị **nhỏ nhất** xuất hiện ở **mọi** hàng của ma trận có các hàng tăng nghiêm ngặt; không có thì `-1`.
2. Dùng hàng đầu tiên làm **tập ứng viên** (đáp án bắt buộc phải nằm trong hàng 0) và kiểm tra bằng `bisect`.
3. Khai thác "tăng nghiêm ngặt" để **đếm**: giá trị nằm trong mọi hàng ⇔ đếm được đúng `m` lần.
4. Biết kỹ thuật **m con trỏ** — tổng quát hóa của "giao hai mảng đã sắp xếp bằng hai con trỏ".
5. Tránh bẫy: đếm khi hàng có trùng lặp (biến thể), trả về giá trị đầu tiên đạt đủ `m` theo **thứ tự duyệt** chứ không phải theo **giá trị**, quên `-1`.

## 2. Đề bài & ví dụ

Cho ma trận `mat` kích thước `m × n`, mỗi hàng được sắp xếp **tăng nghiêm ngặt**. Trả về **phần tử chung nhỏ nhất** của tất cả các hàng. Nếu không có phần tử chung, trả về `-1`.

**Ví dụ 1:** `mat = [[1,2,3,4,5],[2,4,5,8,10],[3,5,7,9,11],[1,3,5,7,9]]` → `5`.

**Ví dụ 2:** `mat = [[1,2,3],[2,3,4],[2,3,5]]` → `2`.

**Ràng buộc:** `1 <= m, n <= 500`; `1 <= mat[i][j] <= 10^4`; mỗi hàng tăng nghiêm ngặt.

In [1]:
import bisect
import random
import time
from collections import Counter
from typing import List

examples = [
    ([[1, 2, 3, 4, 5], [2, 4, 5, 8, 10], [3, 5, 7, 9, 11], [1, 3, 5, 7, 9]], 5),
    ([[1, 2, 3], [2, 3, 4], [2, 3, 5]], 2),
    ([[1, 2], [3, 4]], -1),
]
for mat, want in examples:
    print(mat, "->", want)

[[1, 2, 3, 4, 5], [2, 4, 5, 8, 10], [3, 5, 7, 9, 11], [1, 3, 5, 7, 9]] -> 5
[[1, 2, 3], [2, 3, 4], [2, 3, 5]] -> 2
[[1, 2], [3, 4]] -> -1


## 3. Kiến thức nền

### 3.1 Toán tử `in` trên list và trên set

`x in lst` duyệt tuần tự `O(n)`; `x in st` với `set` trung bình `O(1)` (bảng băm).

In [2]:
row = [2, 4, 5, 8, 10]
st = set(row)
print(5 in row, 6 in row, 5 in st)

True False True


### 3.2 Tìm kiếm nhị phân với `bisect_left`

Trên list tăng dần, `p = bisect_left(a, x)` là vị trí đầu tiên `a[p] >= x`. `x` có mặt ⇔ `p < len(a) and a[p] == x`. Độ phức tạp `O(log n)`.

In [3]:
def contains_sorted(a, x):
    p = bisect.bisect_left(a, x)
    return p < len(a) and a[p] == x


print(contains_sorted([2, 4, 5, 8, 10], 5), contains_sorted([2, 4, 5, 8, 10], 6), contains_sorted([2, 4], 11))

True False False


### 3.3 Đếm bằng mảng tần suất / `Counter`

Giá trị nằm trong `[1, 10^4]` nên có thể dùng mảng `cnt` dài `10^4 + 1`. `collections.Counter` là dict đếm tiện lợi.

In [4]:
cnt = Counter()
for r in [[1, 2, 3], [2, 3, 4], [2, 3, 5]]:
    cnt.update(r)
print(cnt)
print([v for v in sorted(cnt) if cnt[v] == 3])

Counter({2: 3, 3: 3, 1: 1, 4: 1, 5: 1})
[2, 3]


### 3.4 Giao tập hợp `set.intersection`

`set(a) & set(b)` cho phần tử chung. Giao nhiều tập: `set.intersection(*list_of_sets)`. Kết quả không có thứ tự → lấy `min`.

In [5]:
rows = [[1, 2, 3], [2, 3, 4], [2, 3, 5]]
common = set(rows[0]).intersection(*map(set, rows[1:]))
print(common, min(common) if common else -1)

{2, 3} 2


### 3.5 Hai con trỏ giao hai mảng đã sắp xếp

Con trỏ `i` trên `a`, `j` trên `b`: nếu `a[i] < b[j]` thì tăng `i`, nếu lớn hơn thì tăng `j`, bằng nhau thì đó là phần tử chung. Mỗi bước loại bỏ ít nhất một phần tử → `O(len a + len b)`.

In [6]:
def intersect_two(a, b):
    i = j = 0
    out = []
    while i < len(a) and j < len(b):
        if a[i] < b[j]:
            i += 1
        elif a[i] > b[j]:
            j += 1
        else:
            out.append(a[i]); i += 1; j += 1
    return out


print(intersect_two([1, 2, 3, 4, 5], [2, 4, 5, 8, 10]))

[2, 4, 5]


## 4. Ý tưởng nền tảng / trực giác

Với ví dụ 1:

| Ứng viên (hàng 0) | hàng 1 `[2,4,5,8,10]` | hàng 2 `[3,5,7,9,11]` | hàng 3 `[1,3,5,7,9]` | chung? |
|---|---|---|---|---|
| 1 | ✗ | | | không |
| 2 | ✓ | ✗ | | không |
| 3 | ✗ | | | không |
| 4 | ✓ | ✗ | | không |
| 5 | ✓ | ✓ | ✓ | **có → 5** |

Quan sát: (1) đáp án **phải** nằm trong hàng 0; (2) duyệt hàng 0 theo thứ tự **tăng dần**, ứng viên đầu tiên có mặt ở mọi hàng chính là nhỏ nhất → dừng ngay.

In [7]:
mat = examples[0][0]
for v in mat[0]:
    marks = [v in row for row in mat[1:]]
    print(v, marks)
    if all(marks):
        print("=> đáp án", v); break

1 [False, False, True]
2 [True, False, False]
3 [False, True, True]
4 [True, False, False]
5 [True, True, True]
=> đáp án 5


## 5. Cách 1 — Brute Force / baseline

Duyệt `v` trong hàng 0 theo thứ tự; với mỗi hàng khác kiểm tra `v in row` (quét tuần tự). Ứng viên đầu tiên đạt yêu cầu là đáp án.

In [8]:
def brute(mat: List[List[int]]) -> int:
    for v in mat[0]:
        if all(v in row for row in mat[1:]):
            return v
    return -1


def brute_raw(mat):
    """Oracle: thử mọi giá trị từ nhỏ đến lớn trong toàn ma trận."""
    for v in sorted({x for row in mat for x in row}):
        if all(v in row for row in mat):
            return v
    return -1


for mat, want in examples:
    print(brute(mat), brute_raw(mat), want)
    assert brute(mat) == brute_raw(mat) == want

5 5 5
2 2 2
-1 -1 -1


**Độ phức tạp:** worst case `n` ứng viên × `(m−1)` hàng × `O(n)` mỗi lần `in` → `O(m·n^2)` = `500·500^2 = 1.25·10^8` — chậm trong Python. Bộ nhớ `O(1)`.

**Nút thắt:** phép `v in row` không tận dụng việc hàng **đã sắp xếp**.

In [9]:
def worst_case(m, n):
    # m-1 hàng giống hệt hàng 0 = 1..n, hàng cuối chỉ chứa n -> mọi ứng viên đều duyệt gần hết các hàng
    return [list(range(1, n + 1)) for _ in range(m - 1)] + [list(range(n, 2 * n))]


for size in [100, 200, 300]:
    mt = worst_case(size, size)
    t = time.perf_counter(); r = brute(mt); print(size, r, f"{time.perf_counter() - t:.3f}s")

100 100 0.003s
200 200 0.023s
300 300 0.073s


## 6. Bước đột phá

Hai insight độc lập:

1. **Hàng đã sắp xếp ⇒ tìm kiếm nhị phân:** thay `v in row` (`O(n)`) bằng `bisect` (`O(log n)`).
2. **Tăng nghiêm ngặt ⇒ không trùng trong một hàng ⇒ đếm:** số lần `v` xuất hiện trong toàn ma trận **bằng** số hàng chứa `v`. Vậy `v` chung ⇔ `count[v] == m`. Muốn **nhỏ nhất**: duyệt giá trị tăng dần (mảng đếm theo chỉ số), hoặc duyệt ma trận **theo cột** để giá trị nhỏ được đếm sớm.

Quy tắc **m con trỏ**: *nếu* mọi con trỏ chỉ cùng một giá trị *thì* đó là đáp án; *nếu không* thì giá trị lớn nhất hiện tại `hi` là cận dưới của đáp án — dịch mọi con trỏ tới phần tử `≥ hi`.

## 7. Các cách tiếp theo

### Cách 2 — Ứng viên hàng 0 + tìm kiếm nhị phân

In [10]:
def solve_binary(mat: List[List[int]]) -> int:
    for v in mat[0]:
        ok = True
        for row in mat[1:]:
            p = bisect.bisect_left(row, v)
            if p == len(row) or row[p] != v:
                ok = False
                break
        if ok:
            return v
    return -1


for mat, want in examples:
    assert solve_binary(mat) == want
print([solve_binary(m) for m, _ in examples])

[5, 2, -1]


**Độ phức tạp:** `O(n · m · log n)` worst case ≈ `500·500·9 = 2.25·10^6`. Bộ nhớ `O(1)`.

### Cách 3 — Đếm tần suất (mảng đếm theo giá trị)

Đếm mọi phần tử vào `cnt[v]`; sau đó duyệt `v = 1, 2, ...` tìm `cnt[v] == m` đầu tiên.

In [11]:
def solve_count(mat: List[List[int]]) -> int:
    m = len(mat)
    cnt = [0] * 10001
    for row in mat:
        for v in row:
            cnt[v] += 1
    for v in range(1, 10001):
        if cnt[v] == m:
            return v
    return -1


for mat, want in examples:
    assert solve_count(mat) == want
print([solve_count(m) for m, _ in examples])

[5, 2, -1]


**Biến thể dừng sớm — duyệt theo cột:** duyệt `j = 0..n-1`, với mỗi `j` duyệt mọi hàng `i`, tăng `cnt[mat[i][j]]`; ngay khi một giá trị đạt `m` thì trả về. Có an toàn không? Có: nếu `u < v` đều là phần tử chung thì **trong mỗi hàng** `u` đứng ở cột nhỏ hơn `v` (hàng tăng dần), nên cột muộn nhất của `u` (lúc `u` đủ `m`) nhỏ hơn cột muộn nhất của `v`. Vậy giá trị đủ `m` đầu tiên là nhỏ nhất. (Riêng trong cùng một cột hai giá trị không thể cùng hoàn tất, vì mỗi giá trị hoàn tất tại đúng một ô.)

**Độ phức tạp:** `O(m·n + V)` với `V = 10^4` là miền giá trị; bộ nhớ `O(V)`. Bản duyệt theo cột (dưới đây) bỏ được phần `+ V` và thường dừng rất sớm.

In [12]:
def solve_count_columns(mat: List[List[int]]) -> int:
    m, n = len(mat), len(mat[0])
    cnt = [0] * 10001
    for j in range(n):
        for i in range(m):
            v = mat[i][j]
            cnt[v] += 1
            if cnt[v] == m:
                return v
    return -1


for mat, want in examples:
    assert solve_count_columns(mat) == want
print([solve_count_columns(m) for m, _ in examples])

[5, 2, -1]


### Cách 4 — Giao tập hợp

In [13]:
def solve_set(mat: List[List[int]]) -> int:
    common = set(mat[0])
    for row in mat[1:]:
        common &= set(row)
        if not common:
            return -1
    return min(common)


for mat, want in examples:
    assert solve_set(mat) == want
print([solve_set(m) for m, _ in examples])

[5, 2, -1]


**Độ phức tạp:** `O(m·n)` trung bình, bộ nhớ `O(n)`. Không phụ thuộc miền giá trị, không cần hàng tăng nghiêm ngặt (chỉ cần "có mặt").

### Cách 5 — m con trỏ (m-way)

Giữ `ptr[i]` cho từng hàng. Lặp:
- `hi = max(mat[i][ptr[i]])`.
- Với mỗi hàng, dịch `ptr[i]` tới phần tử đầu tiên `≥ hi` (tiến tuần tự). Nếu vượt cuối hàng → `-1`.
- Nếu sau khi dịch, mọi hàng đều có `mat[i][ptr[i]] == hi` → trả `hi`.

In [14]:
def solve_pointers(mat: List[List[int]]) -> int:
    m, n = len(mat), len(mat[0])
    ptr = [0] * m
    while True:
        hi = max(mat[i][ptr[i]] for i in range(m))
        all_equal = True
        for i in range(m):
            while ptr[i] < n and mat[i][ptr[i]] < hi:
                ptr[i] += 1
            if ptr[i] == n:
                return -1
            if mat[i][ptr[i]] != hi:
                all_equal = False
        if all_equal:
            return hi


for mat, want in examples:
    assert solve_pointers(mat) == want
print([solve_pointers(m) for m, _ in examples])

[5, 2, -1]


**Độ phức tạp:** mỗi con trỏ chỉ tiến, nên tổng số bước tiến `≤ m·n`. Mỗi vòng ngoài (trừ vòng cuối) đẩy ít nhất một con trỏ và tốn thêm `O(m)` để tính `hi`, nên worst case `O(m^2·n)` trên lý thuyết; thực tế mỗi vòng thường đẩy nhiều con trỏ cùng lúc. Có thể thay bước tiến tuần tự bằng `bisect` để mỗi lần dịch tốn `O(log n)`. Bộ nhớ `O(m)`.

## 8. TRACE từng bước

Trace Cách 5 trên ví dụ 1:

| vòng | ptr trước | hi | ptr sau | giá trị tại ptr | đều bằng? |
|---|---|---|---|---|---|
| 1 | [0,0,0,0] → (1,2,3,1) | 3 | [2,1,0,2] | (3,4,3,3) | không |
| 2 | → (3,4,3,3) | 4 | [3,1,1,3] | (4,4,5,5) | không |
| 3 | → (4,4,5,5) | 5 | [4,2,1,3] | (5,5,5,5) | **có → 5** |

In [15]:
def trace_pointers(mat):
    m, n = len(mat), len(mat[0])
    ptr = [0] * m
    rnd = 0
    while True:
        rnd += 1
        vals = [mat[i][ptr[i]] for i in range(m)]
        hi = max(vals)
        for i in range(m):
            while ptr[i] < n and mat[i][ptr[i]] < hi:
                ptr[i] += 1
            if ptr[i] == n:
                print(f"vòng {rnd}: hàng {i} hết phần tử -> -1"); return -1
        after = [mat[i][ptr[i]] for i in range(m)]
        print(f"vòng {rnd}: giá trị {vals} hi={hi} -> ptr={ptr} giá trị {after}")
        if all(v == hi for v in after):
            print("=> đáp án", hi); return hi


trace_pointers(examples[0][0])
print()
trace_pointers([[1, 2], [3, 4]])

vòng 1: giá trị [1, 2, 3, 1] hi=3 -> ptr=[2, 1, 0, 1] giá trị [3, 4, 3, 3]
vòng 2: giá trị [3, 4, 3, 3] hi=4 -> ptr=[3, 1, 1, 2] giá trị [4, 4, 5, 5]
vòng 3: giá trị [4, 4, 5, 5] hi=5 -> ptr=[4, 2, 1, 2] giá trị [5, 5, 5, 5]
=> đáp án 5

vòng 1: hàng 0 hết phần tử -> -1


-1

In [16]:
def trace_binary(mat):
    for v in mat[0]:
        info = []
        for r, row in enumerate(mat[1:], 1):
            p = bisect.bisect_left(row, v)
            hit = p < len(row) and row[p] == v
            info.append(f"hàng{r}:p={p}{'✓' if hit else '✗'}")
            if not hit:
                break
        print(f"ứng viên {v}: " + ", ".join(info))
        if len(info) == len(mat) - 1 and info[-1].endswith('✓'):
            return v
    return -1


print("=>", trace_binary(examples[0][0]))

ứng viên 1: hàng1:p=0✗
ứng viên 2: hàng1:p=0✓, hàng2:p=0✗
ứng viên 3: hàng1:p=1✗
ứng viên 4: hàng1:p=1✓, hàng2:p=1✗
ứng viên 5: hàng1:p=2✓, hàng2:p=1✓, hàng3:p=2✓
=> 5


## 9. Code nộp LeetCode

Chọn Cách 3 (đếm) — đơn giản, `O(m·n + 10^4)`. Ghi chú: đúng nhờ mỗi hàng **tăng nghiêm ngặt**.

In [17]:
class Solution:
    def smallestCommonElement(self, mat: List[List[int]]) -> int:
        m = len(mat)
        cnt = [0] * 10001
        for row in mat:
            for v in row:
                cnt[v] += 1
        for v in range(1, 10001):
            if cnt[v] == m:
                return v
        return -1


sol = Solution()
for mat, want in examples:
    print(sol.smallestCommonElement(mat))
    assert sol.smallestCommonElement(mat) == want

5
2
-1


## 10. Vì sao đúng

- **Đếm:** vì mỗi hàng tăng nghiêm ngặt, một giá trị `v` xuất hiện **tối đa một lần** mỗi hàng. Do đó `cnt[v]` = số hàng chứa `v`, và `v` chung ⇔ `cnt[v] = m`. Duyệt `v` tăng dần nên giá trị đầu tiên thỏa là nhỏ nhất. Không có thì `-1`.
- **Ứng viên hàng 0:** mọi phần tử chung đều thuộc hàng 0; hàng 0 tăng dần nên ứng viên đầu tiên thỏa là nhỏ nhất.
- **m con trỏ:** invariant — mọi phần tử chung đều `≥ hi` hiện tại (vì `hi` là phần tử nhỏ nhất còn lại của một hàng nào đó, mọi giá trị nhỏ hơn đã bị hàng đó loại). Dịch các con trỏ qua giá trị `< hi` không bỏ sót phần tử chung nào. Khi mọi con trỏ bằng nhau, đó là phần tử chung nhỏ nhất `≥` mọi giá trị đã loại → nhỏ nhất toàn cục.

## 11. Time / Space Complexity

Gọi `m` hàng, `n` cột, `V = 10^4` (miền giá trị).

| Cách | Time | Space |
|---|---|---|
| 1. Brute (`in` tuần tự) | `O(m·n^2)` | `O(1)` |
| 2. Hàng 0 + bisect | `O(m·n·log n)` | `O(1)` |
| 3. Đếm | `O(m·n + V)` | `O(V)` |
| 4. Giao tập | `O(m·n)` trung bình | `O(n)` |
| 5. m con trỏ | `O(m·n)` bước tiến (+ chi phí tính `hi`) | `O(m)` |

Best case của Cách 1, 2: phần tử đầu hàng 0 là đáp án → rất nhanh.

## 12. Bộ test

In [18]:
solvers = {
    "binary": solve_binary,
    "count": solve_count,
    "count_columns": solve_count_columns,
    "set": solve_set,
    "pointers": solve_pointers,
    "submit": Solution().smallestCommonElement,
}
edge = [
    ([[7]], 7),
    ([[1, 2, 3]], 1),
    ([[5], [5], [5]], 5),
    ([[5], [6]], -1),
    ([[1, 10000], [10000]], 10000),
    ([[1, 3, 5], [2, 4, 6]], -1),
    ([[2, 3, 9], [1, 3, 9], [3, 4, 9]], 3),
]
edge = [(m_, w) for m_, w in edge]
for mat, want in examples + edge:
    assert brute_raw(mat) == want
    for name, f in solvers.items():
        assert f(mat) == want, (name, mat, f(mat), want)
print("ví dụ + edge: OK")

ví dụ + edge: OK


In [19]:
def random_matrix(rng, m, n, V):
    return [sorted(rng.sample(range(1, V + 1), n)) for _ in range(m)]


rng = random.Random(1198)
for trial in range(1500):
    m, n = rng.randint(1, 5), rng.randint(1, 6)
    V = rng.randint(n, n + 6)
    mat = random_matrix(rng, m, n, V)
    want = brute_raw(mat)
    assert brute(mat) == want
    for name, f in solvers.items():
        assert f(mat) == want, (name, mat)
print("1500 test ngẫu nhiên khớp brute force")

1500 test ngẫu nhiên khớp brute force


In [20]:
rng = random.Random(3)
big = random_matrix(rng, 500, 500, 10000)
big_common = [sorted(set(r) | {9999}) [:500] if 9999 in r else sorted(set(r[:-1]) | {9999}) for r in big]
for name, f in solvers.items():
    for mt in (big, big_common):
        t = time.perf_counter(); r = f(mt); dt = time.perf_counter() - t
        print(f"{name:9s} -> {r:5} {dt:.3f}s")
assert len({f(big_common) for f in solvers.values()}) == 1
print("test giới hạn OK")

binary    ->    -1 0.000s
binary    ->  9999 0.001s
count     ->    -1 0.007s
count     ->  9999 0.007s
count_columns ->    -1 0.036s


count_columns ->  9999 0.038s
set       ->    -1 0.000s
set       ->  9999 0.010s
pointers  ->    -1 0.051s
pointers  ->  9999 0.049s
submit    ->    -1 0.008s


submit    ->  9999 0.008s


test giới hạn OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Đếm khi hàng **có thể trùng lặp** (áp dụng mù cho biến thể)

In [21]:
def count_naive(mat):
    m = len(mat)
    cnt = Counter(v for row in mat for v in row)
    cands = [v for v in cnt if cnt[v] >= m]
    return min(cands) if cands else -1


dup = [[1, 1, 2], [2, 3, 4], [2, 5, 6]]    # hàng 0 không tăng nghiêm ngặt
dup2 = [[1, 1, 1], [1, 2, 3], [2, 3, 4]]
print(count_naive(dup), "đúng là", brute_raw(dup))
print(count_naive(dup2), "đúng là", brute_raw(dup2))

2 đúng là 2
1 đúng là -1


**Vì sao sai:** ở `dup2`, giá trị `1` xuất hiện 4 lần nhưng chỉ nằm ở 2 hàng. Đếm số lần ≠ đếm số hàng khi có trùng. **Sửa (cho biến thể):** đếm trên `set(row)` của từng hàng. Bài gốc an toàn vì tăng nghiêm ngặt.

### Lỗi 2 — Chỉ so sánh hàng 0 với hàng 1

In [22]:
def wrong_two_rows(mat):
    if len(mat) == 1:
        return mat[0][0]
    for v in mat[0]:
        if v in mat[1]:
            return v
    return -1


mat = [[1, 2, 3], [1, 3], [3, 4]]
print(wrong_two_rows(mat), "đúng là", brute_raw(mat))

1 đúng là 3


**Vì sao sai:** phải kiểm tra **mọi** hàng. **Sửa:** `all(... for row in mat[1:])`.

### Lỗi 3 — Dùng `bisect` nhưng quên kiểm tra phần tử tìm được có **bằng** `v`

In [23]:
def wrong_bisect_member(mat):
    for v in mat[0]:
        if all(bisect.bisect_left(row, v) < len(row) for row in mat[1:]):   # SAI: chỉ biết có phần tử >= v
            return v
    return -1


mat = [[1, 5], [2, 5], [3, 5]]
print(wrong_bisect_member(mat), "đúng là", brute_raw(mat))

1 đúng là 5


**Vì sao sai:** `bisect_left(row, v) < len(row)` chỉ nói "hàng có phần tử `≥ v`", không nói `v` có mặt. Với `v = 1`, mọi hàng đều có phần tử `≥ 1` nên trả nhầm `1`. **Sửa:** `p < len(row) and row[p] == v`.

### Lỗi 4 — Dùng `set` giao rồi trả `common.pop()` thay vì `min(common)`

In [24]:
def wrong_pop(mat):
    common = set(mat[0])
    for row in mat[1:]:
        common &= set(row)
    return common.pop() if common else -1


mat = [[3, 70, 100], [3, 70, 100], [3, 70, 100]]   # tình cờ đúng
mat2 = [[1, 8], [1, 8]]                            # 8 % 8 == 0 nên 8 đứng trước 1 trong bảng băm
print(wrong_pop(mat), wrong_pop(mat2), "| đúng là", brute_raw(mat), brute_raw(mat2))
print("thứ tự lặp của set {1, 8}:", list({1, 8}))

3 8 | đúng là 3 1
thứ tự lặp của set {1, 8}: [8, 1]


**Vì sao sai:** `set` không có thứ tự đảm bảo; `pop()` trả một phần tử **bất kỳ**. Với số nguyên CPython băm theo chính giá trị nên nhiều test nhỏ "trông như" có thứ tự (ví dụ đầu tình cờ đúng), nhưng `{1, 8}` lặp ra `8` trước `1` (vì `8 % 8 = 0`) → trả nhầm `8`. **Sửa:** `min(common)`.

## 14. Mẹo & nhận dạng pattern

- **"Phần tử chung của nhiều dãy đã sắp xếp"** → hai/m con trỏ, hoặc bisect từ dãy ngắn nhất.
- **"Mỗi dãy không trùng"** → đếm số lần = đếm số dãy chứa.
- **Miền giá trị nhỏ (≤ 10^4)** → mảng đếm thay cho dict, duyệt theo giá trị để lấy min/max.
- **Đáp án phải thuộc một tập cụ thể (hàng 0)** → dùng tập đó làm ứng viên, chọn tập **nhỏ nhất** nếu các hàng khác độ dài.
- `set` không có thứ tự — muốn nhỏ nhất thì `min`.

## 15. Biến thể

1. **LeetCode 349 / 350 — Intersection of Two Arrays (I, II).**
2. **LeetCode 2248 — Intersection of Multiple Arrays:** trả mọi phần tử chung đã sắp xếp.
3. **Hàng có thể trùng lặp** → đếm theo `set(row)`.
4. **Hàng không sắp xếp** → giao tập hợp hoặc đếm theo `set`.
5. **Ma trận cực lớn, hàng dài khác nhau** → chọn hàng ngắn nhất làm ứng viên, bisect các hàng khác.

In [25]:
def common_all_sorted(mat):
    """Biến thể 2/3: mọi phần tử chung, hàng có thể trùng / không sắp xếp."""
    cnt = Counter()
    for row in mat:
        cnt.update(set(row))
    return sorted(v for v, c in cnt.items() if c == len(mat))


print(common_all_sorted([[3, 1, 2, 4, 5], [1, 2, 3, 4], [3, 4, 5, 6]]), common_all_sorted([[1, 1, 2], [1, 2, 2], [2, 1]]))

[3, 4] [1, 2]


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute | Ứng viên hàng 0, `in` tuần tự | `O(m·n^2)` | `O(1)` | Oracle |
| 2. Bisect | Ứng viên hàng 0, tìm nhị phân | `O(m·n·log n)` | `O(1)` | Hàng sắp xếp, không muốn tốn bộ nhớ |
| 3. Đếm | `cnt[v] == m` (duyệt giá trị hoặc theo cột) | `O(m·n + V)` | `O(V)` | Miền giá trị nhỏ, hàng không trùng |
| 4. Giao tập | `&=` từng hàng | `O(m·n)` | `O(n)` | Tổng quát, hàng không cần sắp xếp |
| 5. m con trỏ | Dịch về `hi` | `O(m·n)`+ | `O(m)` | Dữ liệu dạng luồng (stream) đã sắp xếp |

## 17. Bài tập vận dụng

**Bài 1 (đoán output):** `smallestCommonElement([[1,4,7],[4,7,9],[0,4,7]])`? (giả sử cho phép 0)

**Bài 2 (tính tay):** trace m con trỏ với `[[2,6,8],[1,6,8],[6,7,8]]`.

**Bài 3 (điền TODO):** hoàn thiện `intersect_sorted_k(lists)` trả **mọi** phần tử chung (tăng dần) của k list đã sắp xếp nghiêm ngặt.

**Bài 4 (tự viết lại):** viết Cách 2 nhưng chọn **hàng ngắn nhất** làm ứng viên (hàng có thể có độ dài khác nhau).

**Bài 5 (mini challenge):** trả về phần tử **xuất hiện ở nhiều hàng nhất**; hòa thì lấy giá trị nhỏ nhất.

In [26]:
def intersect_sorted_k(lists):
    # TODO
    pass


def smallest_common_shortest_row(mat):
    # TODO: hàng có độ dài khác nhau
    raise NotImplementedError


def most_frequent_across_rows(mat):
    # TODO
    raise NotImplementedError

# assert intersect_sorted_k([[1, 2, 3, 5], [2, 3, 5, 7], [0, 2, 5]]) == [2, 5]
# assert smallest_common_shortest_row([[1, 2, 3, 4, 5], [5], [2, 5]]) == 5

## 18. Gợi ý 3 mức

- **Mức 1:** Đáp án phải nằm trong hàng 0.
- **Mức 2:** Mỗi hàng không có trùng → số lần xuất hiện của `v` = số hàng chứa `v`.
- **Mức 3:** Bài 3: chạy m con trỏ nhưng thay vì trả về, ghi nhận `hi` rồi tăng mọi con trỏ thêm 1 và tiếp tục. Bài 5: `Counter` trên `set(row)`, chọn theo khóa `(-count, value)`.

## 19. Đáp án tham khảo

**Bài 1:** `4`.

**Bài 2:** vòng 1: giá trị `(2,1,6)`, `hi = 6` → mọi con trỏ dịch tới `6` → `(6,6,6)` → đáp án `6`.

In [27]:
assert solve_set([[1, 4, 7], [4, 7, 9], [0, 4, 7]]) == 4
assert solve_pointers([[2, 6, 8], [1, 6, 8], [6, 7, 8]]) == 6


def intersect_sorted_k(lists):
    k = len(lists)
    ptr = [0] * k
    out = []
    while all(ptr[i] < len(lists[i]) for i in range(k)):
        hi = max(lists[i][ptr[i]] for i in range(k))
        ok = True
        for i in range(k):
            while ptr[i] < len(lists[i]) and lists[i][ptr[i]] < hi:
                ptr[i] += 1
            if ptr[i] == len(lists[i]) or lists[i][ptr[i]] != hi:
                ok = False
        if ok:
            out.append(hi)
            ptr = [p + 1 for p in ptr]
    return out


def smallest_common_shortest_row(mat):
    base = min(mat, key=len)
    for v in base:
        if all((lambda r: (lambda p: p < len(r) and r[p] == v)(bisect.bisect_left(r, v)))(row) for row in mat):
            return v
    return -1


def most_frequent_across_rows(mat):
    cnt = Counter()
    for row in mat:
        cnt.update(set(row))
    return min(cnt, key=lambda v: (-cnt[v], v))


assert intersect_sorted_k([[1, 2, 3, 5], [2, 3, 5, 7], [0, 2, 5]]) == [2, 5]
assert smallest_common_shortest_row([[1, 2, 3, 4, 5], [5], [2, 5]]) == 5
assert most_frequent_across_rows([[1, 2], [2, 3], [3, 4], [3]]) == 3
rng = random.Random(19)
for _ in range(800):
    mat = [sorted(rng.sample(range(1, 12), rng.randint(1, 6))) for _ in range(rng.randint(1, 4))]
    common = sorted(set(mat[0]).intersection(*map(set, mat[1:])))
    assert intersect_sorted_k(mat) == common
    assert smallest_common_shortest_row(mat) == (common[0] if common else -1)
print("Đáp án bài tập: OK")

Đáp án bài tập: OK


## 20. Template nhớ nhanh + Checklist trước khi submit

```python
# Đếm (hàng tăng nghiêm ngặt, giá trị nhỏ)
cnt = [0] * (MAXV + 1)
for row in mat:
    for v in row: cnt[v] += 1
for v in range(1, MAXV + 1):
    if cnt[v] == len(mat): return v
return -1

# Ứng viên hàng 0 + bisect
for v in mat[0]:
    if all((p := bisect_left(r, v)) < len(r) and r[p] == v for r in mat[1:]): return v
```

**Checklist:**
- [ ] Trả `-1` khi không có phần tử chung.
- [ ] Ma trận 1 hàng → trả `mat[0][0]`.
- [ ] Đếm chỉ đúng khi mỗi hàng **không trùng** (đọc kỹ "strictly increasing").
- [ ] Lấy **nhỏ nhất theo giá trị** (duyệt theo cột chỉ đúng vì hàng tăng dần); dùng `min`, không `set.pop()`.
- [ ] `bisect`: kiểm tra `p < len(row)` trước khi đọc `row[p]`.